# **1) PIP INSTALL**

In [1]:
!pip install datasets tensorflow -q

# **2) IMPORT LIBRARIES**

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import os
import random

# **3) SET RANDOM SEED**

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random Seed Set:", SEED)

Random Seed Set: 42


# **4) LOAD DATASET**

In [6]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


# **5) EXTRACT STORIES**

In [8]:
stories = []

# Take first 1000 stories
for item in ds['train'].select(range(1000)):
    stories.append(item['text'])

print("Total Stories:", len(stories))
print("\nSample Story:\n")
print(stories[0][:1000])

Total Stories: 1000

Sample Story:

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


# **6) COMBINE TEXT**

In [9]:
text_data = " ".join(stories)

print("Total Characters:", len(text_data))

Total Characters: 942639


# **7) TOKENIZATION**

In [10]:
vocab_size = 5000

# Create tokenizer

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts([text_data])

# Convert text to sequence
sequence_data = tokenizer.texts_to_sequences([text_data])[0]

print("Total Tokens:", len(sequence_data))

Total Tokens: 183978


# **8) CREATE INPUT SEQUENCES**

In [11]:
sequence_length = 20

input_sequences = []

for i in range(sequence_length, len(sequence_data)):
    seq = sequence_data[i-sequence_length:i+1]
    input_sequences.append(seq)

input_sequences = np.array(input_sequences)

print("Shape:", input_sequences.shape)

Shape: (183958, 21)


# **9) SPLIT X AND y**

In [12]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (183958, 20)
y Shape: (183958,)


# **10) BUILD GRU MODEL**

In [13]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_shape=(sequence_length,)
    )
)

# First GRU
model.add(GRU(128, return_sequences=True))
model.add(Dropout(0.2))

# Second GRU
model.add(GRU(128))

# Output Layer
model.add(Dense(vocab_size, activation='softmax'))

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Show summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 128)        │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5000)           │       645,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,483,144 (5.66 MB)

 Trainable params: 1,483,144 (5.66 MB)

 Non-trainable params: 0 (0.00 B)